# PAIRCROP–CE — one arm, two T4 GPUs, 300 updates
Người dùng tự chạy. Notebook chưa được chạy bởi tác giả.
Hai nhánh mới cùng baseline/CE/schedule/initialization; khác nguồn patch và tọa độ patch tương ứng.
14fit,7exposeddevelopment,8new-to-adaptation probe(4skate+4surf), không phải test độc lập toàn hệ thống.
FULLFRAME chạy trước: tạo cachefull/crop/G chung và chặn train nếu featurecompatibility thấtbại.
PAIRCROP chạy sau: mount nguyên folder shared, không tải CLIP/trích feature lại.
KhôngCROSS/SCST/routing. Không dùng adapterCE/CROSS cũ hay checkpointF. BậtInternet, chọnT4×2,Save&RunAll.


In [ ]:
print('PAIR SOURCE PAIRCROP: START',flush=True)
from pathlib import Path
VARIANT='PAIRCROP'

CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/qformer-scst-object-w05-legacy/qformer_scst_object_w05_legacy.pth')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')

# SỬA thành folder chứa shared_receipt.json và feature_cache.bin từ notebook FULLFRAME đã chạy xong.
SHARED=Path('/kaggle/input/datasets/ducanh2403/pair-crop-shared/pair_crop_shared')
COCO_IMAGES=SHARED/'images'
VISUAL_CACHE=Path('/tmp/not_used_in_paircrop')

OUTPUT=Path('/kaggle/working/pair_source_'+VARIANT.lower()+'_ce')


In [ ]:
import base64,hashlib,json,os,shutil,subprocess,sys,zlib
os.environ.update(PYTHONUNBUFFERED='1',HDF5_USE_FILE_LOCKING='FALSE',CUBLAS_WORKSPACE_CONFIG=':4096:8',PYTHONHASHSEED='42',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',TOKENIZERS_PARALLELISM='false')
REPO=Path('/tmp/Image_Captioning-pair-source-'+VARIANT.lower());LOG=Path('/tmp/pair_source_'+VARIANT.lower()+'.log');CONFIG=REPO/'pair_config.json'
def run(args,cwd=None):
    args=list(map(str,args));print('Running:',' '.join(args),flush=True)
    with LOG.open('a',encoding='utf-8') as log:
        process=subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        code=process.wait()
    if code:raise subprocess.CalledProcessError(code,args)
for p in (CHECKPOINT,COCO_JSON):
    if not p.is_file():raise FileNotFoundError(p)
if VARIANT=='FULLFRAME':
    assert VISUAL_CACHE.is_dir() and COCO_IMAGES.is_dir()
    assert not SHARED.exists(),'Use fresh shared folder/session'
else:
    for name in ('feature_cache.bin','shared_receipt.json','fullframe_complete_rank0.json','fullframe_protocol.json'):
        assert (SHARED/name).is_file(),f'Wrong shared path or unfinished FULLFRAME: {SHARED/name}'
assert not OUTPUT.exists(),'Use a fresh OUTPUT; no overwrite'
import torch
assert torch.cuda.device_count()==2 and all(torch.cuda.get_device_name(i)=='Tesla T4' for i in (0,1)),'Select T4 x2'
if not REPO.exists():run(['git','clone','--no-checkout','https://github.com/Supzxjee/Image_Captioning.git',REPO])
run(['git','fetch','--depth','1','origin','6e2f9ef8553c7a9637916a7cf0ec3a90697b4c33'],REPO);run(['git','checkout','--detach','6e2f9ef8553c7a9637916a7cf0ec3a90697b4c33'],REPO)
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','numpy','Pillow'])
PAYLOAD='...'
raw=zlib.decompress(base64.b64decode(PAYLOAD,validate=True));assert hashlib.sha256(raw).hexdigest()=='c2296c7ec6b2afef0b676637a9e0a701a72e70a6718c0d63b99de879a55f3ae4'
packet=json.loads(raw);hashes={'phrase_data.py': '3d076b5257ad18c295d29607efb3cb81605d28075751073a40b3e3ae4d6e81be', 'phrase_model.py': 'cb3dba8f6e71698db9213f1dbd7137f8c5a0c3613d76765bcadb263ec4844834', 'run_phrase.py': 'd615ab32d64ff98d3d542124094f2cab8fedd59b3cb8fbb7d5ebc2bdf4c25997', 'test_phrase_model.py': 'a875e25eb7022108ac8105f709c6cd560fca90cd4b95787c798c1a020e6a611d', 'pair_data.py': '5c36012b668c93fcbe5d1d94bdaa8d6267eda31ad6a2ba9ad3f3b88a175ba89f', 'pair_model.py': '7b88488041f37f92ad0c1c06dbe8a980c591d2cf898efaf9b7270074286c0e74', 'run_pair.py': 'a187ff2fbaea618169e7e888a0e34fc9e32fc909cb7c51cf22f18cb39b8a87a2', 'test_pair_model.py': 'd714e6f9087fa75770b5cd42558f37bd99ec4c8299e7d8633116289a282d52d5', 'crop_geometry.py': '2fa29a4ce91342c63a36b9d6ef00884679d39d73245cc2b572a183ecc7193b80', 'test_preparation.py': '4a91df7286a3fc472265c28c452f747627984e07a88a308a8087a5f2db6f4981', 'test_pair_data.py': 'fded5ae16971931c85701ffdd57f7ea4edd26368eb02381838a8f544b861506a', 'inspect_visual_metadata.py': 'c72bfb01e72049dc811c8d3381a2e3a4423e5bf66cad6fe5ed21d8af2a633b72', 'audit_cache_metadata.py': '6fefa2cffffe1e03d14f65c3b9b62a1acf62644c0af133bf506dd4641a62eb98'}
for name,source in packet['sources'].items():
    assert hashlib.sha256(source.encode()).hexdigest()==hashes[name]
    (REPO/name).write_text(source,encoding='utf-8')
manifest=REPO/'locked_manifest.json';manifest.write_text(packet['manifest'],encoding='utf-8')
cfg=dict(arm='CE',variant=VARIANT,checkpoint=str(CHECKPOINT),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),coco_images=str(COCO_IMAGES),
    shared=str(SHARED),output=str(OUTPUT),manifest=str(manifest),manifest_sha256=hashlib.sha256(packet['manifest'].encode()).hexdigest(),source_sha256=hashes,project_commit='6e2f9ef8553c7a9637916a7cf0ec3a90697b4c33')
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
run([sys.executable,'-u','test_preparation.py'],REPO)
run([sys.executable,'-u','test_pair_data.py'],REPO)
run([sys.executable,'-u','test_pair_model.py'],REPO)
run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0','run_pair.py','smoke','--config',CONFIG],REPO)


In [ ]:
try:
    run([sys.executable,'-u','run_pair.py','prepare','--config',CONFIG],REPO)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0','run_pair.py','worker','--config',CONFIG],REPO)
    run([sys.executable,'-u','run_pair.py','finish','--config',CONFIG],REPO)
finally:
    if OUTPUT.is_dir():
        shutil.copy2(LOG,OUTPUT/'run.log')
        for name in ('compatibility.json','original_metadata.json'):
            if (SHARED/name).is_file():shutil.copy2(SHARED/name,OUTPUT/name)
        run([sys.executable,'-u','run_pair.py','archive','--config',CONFIG],REPO)
print('SEND AFTER BOTH ARMS:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP adapter separately:',OUTPUT/'adaptation.bin',flush=True)
if VARIANT=='FULLFRAME':print('NEXT: preserve/upload WHOLE shared folder including images:',SHARED,flush=True)


## Kết quả và bước tiếp
FULLFRAME: lưu nguyên /kaggle/working/pair_crop_shared thành dataset (feature_cache.bin,receipt,images và fullframe_* references).
Dữ liệu nhỏ, chỉ46ảnh/58cropviews; không copy hay mã hóa lại123287ảnh.
Sửa SHARED trong PAIRCROP thành mountthực rồi chạy; đừng thay treatment sau khi xem outputprobe của control.
Gửi cả pair_source_fullframe_ce_review.zip và pair_source_paircrop_ce_review.zip. ZIP review không chứa weights/features/images;
shared folder giữ ảnh để tạo review mù sau khi cảhai hoàn tất. Giữ adaptation.bin riêng, không dùng như checkpoint caption thường.
Primary metric cần chấm mù ngữ nghĩa sinh tựdo của đúng cặp. Ranking/xácsuất/exactmatch không thay semanticaccuracy.
Nếu featuregate fail, gửi FULLFRAME reviewZIP + pair_crop_shared/compatibility.json; không bỏassert/nới tolerance.
